In [0]:
# #key cols list
# key_cols="['airport_id']"
# key_cols_list=eval(key_cols)

# #cdc column
# cdc_col="modified_date"

# #Backdated Refresh
# backdated_refresh=""

# #source object
# source_object="silver_airports"

# #source schema
# source_schema="silvers"

# target_schema="golds"

# target_object="dim_airports"

# surrogate_key="DimAirportkey"

# catalog="workspace"

In [0]:
#key cols list
key_cols="['passenger_id']"
key_cols_list=eval(key_cols)

#cdc column
cdc_col="modified_date"

#Backdated Refresh
backdated_refresh=""

#source object
source_object="silver_passengers"

#source schema
source_schema="silvers"

target_schema="golds"

target_object="dim_passengers"

surrogate_key="Dimpassengerkey"

catalog="workspace"

In [0]:
key_cols_list

['passenger_id']

### Incremental Data Ingestion

#### Last Load Date

In [0]:
if len(backdated_refresh)==0:
    if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
        last_load=spark.sql(f"select max({cdc_col}) from {catalog}.{target_schema}.{target_object}").collect()[0][0]
    else:
        last_load="1900-01-01 00:00:00"
else:
    last_load=backdated_refresh

last_load

datetime.datetime(2026, 3, 10, 19, 29, 9, 880000)

In [0]:
last_load

datetime.datetime(2026, 3, 10, 19, 29, 9, 880000)

In [0]:
## filter only new data

df_src=spark.sql(f"select * from {source_schema}.{source_object} where {cdc_col}>='{last_load}'")

In [0]:
df_src.display()

passenger_id,name,gender,nationality,modified_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z


In [0]:
if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    key_cols_string_incremental=",".join(key_cols_list)
    df_trg=spark.sql(f"select {key_cols_string_incremental},{surrogate_key},create_date,update_date from {catalog}.{target_schema}.{target_object}")
else:
    key_cols_string_init=[f"'' as {i}" for i in key_cols_list]
    key_cols_string_init=",".join(key_cols_string_init)

    df_trg=spark.sql(f" select {key_cols_string_init},cast('0' as int) as {surrogate_key},cast('1900-01-01 00:00:00' as timestamp) as create_date,cast('1900-01-01 00:00:00' as timestamp) as update_date ")

In [0]:
df_trg.display()

passenger_id,Dimpassengerkey,create_date,update_date
P0001,1,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0002,2,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0003,3,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0004,4,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0005,5,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0006,6,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0007,7,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0008,8,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0009,9,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0010,10,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z


In [0]:
join_condition='AND'.join([f"src.{i}=trg.{i}" for i in key_cols_list])

df_src.createOrReplaceTempView("src")
df_trg.createOrReplaceTempView("trg")

df_join=spark.sql(f"""
                  select src.*,
                         trg.{surrogate_key},
                         trg.create_date,
                         trg.update_date
                  from src 
                  left join trg 
                  on {join_condition}
                  """)
df_join.display()

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z,1,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z,2,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z,3,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z,4,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z,5,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z,6,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z,7,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z,8,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z,9,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z,10,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import*
df_old = df_join.filter(col(f'{surrogate_key}').isNotNull())
df_old.display()


passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z,1,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z,2,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z,3,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z,4,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z,5,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z,6,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z,7,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z,8,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z,9,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z,10,2026-03-11T02:28:24.723Z,2026-03-11T02:28:24.723Z


In [0]:
df_new = df_join.filter(col(f'{surrogate_key}').isNull())   
df_new.display()

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0221,Amy Welch,Male,Croatia,2026-03-11T02:46:40.344Z,null,null,null
P0222,Maria Taylor,Male,Lao People's Democratic Republic,2026-03-11T02:46:40.344Z,null,null,null
P0223,Nicholas Gomez,Female,Cook Islands,2026-03-11T02:46:40.344Z,null,null,null
P0224,Jason Jensen,Female,Rwanda,2026-03-11T02:46:40.344Z,null,null,null
P0225,William Lopez,Male,Heard Island and McDonald Islands,2026-03-11T02:46:40.344Z,null,null,null


In [0]:
###preparing df_old

df_old_enr=df_old.withColumn('update_date',current_timestamp())
df_old_enr.display()

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z,1,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z,2,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z,3,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z,4,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z,5,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z,6,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z,7,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z,8,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z,9,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z,10,2026-03-11T02:28:24.723Z,2026-03-11T02:49:52.476Z


In [0]:
### Preparing df_new

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    max_surrogate_key=spark.sql(f"""
                                select max({surrogate_key}) from {catalog}.{target_schema}.{target_object}""").collect()[0][0]
    df_new_enr=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
                      .withColumn('create_date',current_timestamp())\
                      .withColumn('update_date',current_timestamp())

else:
    max_surrogate_key=0
    df_new_enr=df_new.withColumn(f'{surrogate_key}',lit(max_surrogate_key)+lit(1)+monotonically_increasing_id())\
                     .withColumn('create_date',current_timestamp())\
                     .withColumn('update_date',current_timestamp())
                
df_new_enr.display()

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0221,Amy Welch,Male,Croatia,2026-03-11T02:46:40.344Z,221,2026-03-11T02:49:54.092Z,2026-03-11T02:49:54.092Z
P0222,Maria Taylor,Male,Lao People's Democratic Republic,2026-03-11T02:46:40.344Z,222,2026-03-11T02:49:54.092Z,2026-03-11T02:49:54.092Z
P0223,Nicholas Gomez,Female,Cook Islands,2026-03-11T02:46:40.344Z,223,2026-03-11T02:49:54.092Z,2026-03-11T02:49:54.092Z
P0224,Jason Jensen,Female,Rwanda,2026-03-11T02:46:40.344Z,224,2026-03-11T02:49:54.092Z,2026-03-11T02:49:54.092Z
P0225,William Lopez,Male,Heard Island and McDonald Islands,2026-03-11T02:46:40.344Z,225,2026-03-11T02:49:54.092Z,2026-03-11T02:49:54.092Z


In [0]:
### old record union new record

df_union=df_old_enr.unionByName(df_new_enr)
df_union.display()

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z,1,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z,2,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z,3,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z,4,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z,5,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z,6,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z,7,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z,8,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z,9,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z,10,2026-03-11T02:28:24.723Z,2026-03-11T02:49:55.065Z


In [0]:
from delta.tables import DeltaTable

if spark.catalog.tableExists(f"{catalog}.{target_schema}.{target_object}"):
    dlt_obj = DeltaTable.forName(spark, f"{catalog}.{target_schema}.{target_object}")
    dlt_obj.alias("trg")\
           .merge(df_union.alias("src"), f"trg.{key_cols_list[0]}=src.{key_cols_list[0]}")\
           .whenMatchedUpdateAll(condition=f"src.{cdc_col}>=trg.{cdc_col}")\
           .whenNotMatchedInsertAll()\
           .execute()
else:
    df_union.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"{catalog}.{target_schema}.{target_object}")

In [0]:
%sql
select * from golds.dim_passengers

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0001,Kevin Ferguson,Male,Reunion,2026-03-11T02:46:40.344Z,1,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0002,Kathleen Martinez DVM,Female,Burkina Faso,2026-03-11T02:46:40.344Z,2,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0003,Cynthia Frazier,Male,Marshall Islands,2026-03-11T02:46:40.344Z,3,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0004,Ryan Ramsey,Male,Niger,2026-03-11T02:46:40.344Z,4,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0005,Mike Kim,Male,Taiwan,2026-03-11T02:46:40.344Z,5,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0006,Diana Adams,Male,Mayotte,2026-03-11T02:46:40.344Z,6,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0007,Sharon Moon,Male,Madagascar,2026-03-11T02:46:40.344Z,7,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0008,Cheryl Glenn,Male,Maldives,2026-03-11T02:46:40.344Z,8,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0009,Allen Lowery,Male,Rwanda,2026-03-11T02:46:40.344Z,9,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
P0010,Maria Medina,Male,Denmark,2026-03-11T02:46:40.344Z,10,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z


In [0]:
%sql
select * from workspace.golds.dim_passengers where passenger_id='P0049'

passenger_id,name,gender,nationality,modified_date,Dimpassengerkey,create_date,update_date
P0049,Justin Thomas,Female,Tokelau,2026-03-11T02:46:40.344Z,49,2026-03-11T02:28:24.723Z,2026-03-11T02:49:56.416Z
